In [1]:
import os, gc, sys, shutil, subprocess, joblib, chromadb, logging
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

os.environ["NUMBA_NUM_THREADS"] = str(os.cpu_count())
os.environ["NUMBA_THREADING_LAYER"] = "tbb"

import numpy as np
import polars as pl
from collections import Counter

from pathlib import Path

from dotenv import load_dotenv

from tqdm.auto import tqdm

from pacmap import PaCMAP
from hdbscan import HDBSCAN
from sklearn.neighbors import KNeighborsClassifier

import torch
from sentence_transformers import SentenceTransformer

from transformers import (
    BitsAndBytesConfig, AutoTokenizer, 
    AutoModelForCausalLM
)

from transformers import logging as hf_logging
from transformers.utils.logging import disable_progress_bar

In [2]:
# Define data paths and verify chromadb data integrity
TRAIN_PATH = Path("../data/train.csv")
TEST_PATH = Path("../data/test.csv")

CHROMA_DB_DATA = Path("data/vector_store/knowledge_db")
CHROMA_DB_PATH = Path("data/vector_store/knowledge_db")
CHROMA_DB_NAME = "knowledge_db"

# if CHROMA_DB_PATH.exists():
#     shutil.rmtree(CHROMA_DB_PATH)

# shutil.copytree(CHROMA_DB_DATA, CHROMA_DB_PATH)

# print("Copied files:\n")
# for root, _, files in os.walk(CHROMA_DB_PATH):
#     for f in files:
#         p = os.path.join(root, f)
#         print(f"{p}  ({os.path.getsize(p)} bytes)")

_verify_client = chromadb.PersistentClient(path=CHROMA_DB_PATH)
_verify_collection = _verify_client.get_collection(name=CHROMA_DB_NAME)

_count = _verify_collection.count()
assert _count > 0, "Collection is empty after copying from the Kaggle dataset"

_check = _verify_collection.get(limit=5, include=["embeddings"])
assert len(_check["ids"]) == 5, "Could not retrieve embeddings - vector segment may not have been copied"

_emb = np.array(_check["embeddings"][0], dtype=np.float32) # type: ignore
_sanity = _verify_collection.query(query_embeddings=[_emb.tolist()], n_results=1)
assert _sanity["ids"][0][0] == _check["ids"][0], \
    "Query did not return the vector's own nearest neighbor - index looks corrupted"

print(f"\nChroma DB copy verified: {_count} vectors present and searchable.")

del _verify_client, _verify_collection, _check, _emb, _sanity
gc.collect()


Chroma DB copy verified: 12405 vectors present and searchable.


96

In [3]:
# Configure HuggingFace API Key
load_dotenv()

os.environ["HF_TOKEN"] = os.getenv("HF_READ_TOKEN") if os.getenv("HF_READ_TOKEN") else "" # type: ignore

# Configure logging levels to hide model-loading report
logging.getLogger("transformers").setLevel(logging.WARNING)
logging.getLogger("huggingface_hub").setLevel(logging.WARNING)
logging.getLogger("sentence_transformers").setLevel(logging.WARNING)

hf_logging.set_verbosity_error()

disable_progress_bar()

In [4]:
# Define quantization config, models, model loading functions, and intermediate file paths
bnb_config = BitsAndBytesConfig(
    load_in_8bit=True,
    llm_int8_threshold=6.0,
    llm_int8_has_fp16_weight=False,
    # llm_int8_enable_fp32_cpu_offload=True
)

model_config = {
    "embedder_model": "Qwen/Qwen3-Embedding-4B",
    "reranker_model": "Qwen/Qwen3-Reranker-8B",
    "generative_slm": "Qwen/Qwen2.5-14B-Instruct"
}

local_model_config = {
    "embedder_model": Path("../local_model/qwen3-embedding-4b"),
    "reranker_model": Path("../local_model/qwen3-reranker-8B"),
    "generative_slm": Path("../local_model/qwen2.5-14b-instruct")
}

for path in local_model_config.values():
    path.mkdir(parents=True, exist_ok=True)

# Define functions to load models from local cache if present
def load_embedder(local_path: Path, hf_id: str):
    path_str = str(local_path)
    is_saved = (local_path / "modules.json").exists()
    model_name_or_path = path_str if is_saved else hf_id
    
    print(f"{'Loading' if is_saved else 'Downloading'} Embedder: {model_name_or_path}")
    
    model = SentenceTransformer(
        model_name_or_path,
        model_kwargs={
            "trust_remote_code": True,
            "dtype": torch.float16,
            "device_map": "cuda:0"
        },
        processor_kwargs={"padding_side": "left"},
        prompts={"mcq_query": f"Instruct: {MCQ_EMBED_INSTRUCTION}\nQuery: "},
        default_prompt_name="mcq_query"
    )
    
    if not is_saved:
        model.save(path_str)
        
    return model

def load_slm(local_path: Path, hf_id: str):
    path_str = str(local_path)
    is_saved = (local_path / "config.json").exists()
    model_name_or_path = path_str if is_saved else hf_id
    
    print(f"{'Loading' if is_saved else 'Downloading'} SLM: {model_name_or_path}")
    
    tokenizer = AutoTokenizer.from_pretrained(
        model_name_or_path, 
        trust_remote_code=True,
        truncation_side="left",
        padding_side="left"
    )
    
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
        
    model = AutoModelForCausalLM.from_pretrained(
        model_name_or_path,
        quantization_config=bnb_config,
        attn_implementation="sdpa",
        trust_remote_code=True,
        dtype=torch.float16, 
        device_map="auto"
    )
    
    model.config.use_cache = False
    model.eval()
    
    if not is_saved:
        tokenizer.save_pretrained(path_str)
        model.save_pretrained(path_str)
        
    return tokenizer, model

inter_path_config = {
    "working_dir": Path.cwd(),
    
    "train_retrieved": Path("train_retrieved.parquet"),
    "test_retrieved": Path("test_retrieved.parquet"),
    "train_ranked": Path("train_ranked.parquet"),
    "test_ranked": Path("test_ranked.parquet"),
    "train_final": Path("train_final.parquet"),
    "test_final": Path("test_final.parquet"),

    "submission": Path("submission.csv")
}

In [ ]:
# Define seeds, option cols, and model instructions
SEED = 42
OPTION_COLS = ["A", "B", "C", "D", "E"]

def reset_gpu() -> None:
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

MCQ_EMBED_INSTRUCTION = (
    "Given a multiple-choice science question and its answer options, retrieve document "
    "chunks that contain the information needed to determine the correct answer"
)

MCQ_RERANK_INSTRUCTION = (
    "Given a multiple-choice question and its answer options, judge whether this "
    "chunk contains information that helps determine the correct answer"
)

GENERATIVE_RANKING_INSTRUCTION = (
    "You are an expert scientist. Use the provided context to answer the multiple-choice "
    "question. If the context is irrelevant or conflicts with what you are confident is "
    "correct, rely on your own knowledge instead. "
    "Respond with only the letter of the single best option. Do not include any other text."
)

GENERATIVE_SLM_INSTRUCTION = (
    "You are an expert scientist. Use the provided context to answer the multiple-choice "
    "question. If the context is irrelevant or conflicts with what you are confident is "
    "correct, rely on your own knowledge instead. "
    "Respond with the letters of the three most likely options, best first, separated by "
    "spaces (e.g. \"B D A\"). Do not include any other text."
)

In [ ]:
# Read train and test data, construct `mcq_query` and initialize chromadb client
train_data = pl.read_csv(TRAIN_PATH)
test_data = pl.read_csv(TEST_PATH)

train_data = train_data.with_columns(
    (
        pl.lit("Prompt : ") + pl.col("prompt")
        + pl.lit(" Options: ")
        + pl.concat_str(
            [pl.lit(f"{opt}) ") + pl.col(opt).fill_null(" ") for opt in OPTION_COLS],
            separator=" "
        )
    ).alias("mcq_query")
)

test_data = test_data.with_columns(
    (
        pl.lit("Prompt : ") + pl.col("prompt")
        + pl.lit(" Options: ")
        + pl.concat_str(
            [pl.lit(f"{opt}) ") + pl.col(opt).fill_null(" ") for opt in OPTION_COLS],
            separator=" "
        )
    ).alias("mcq_query")
)

chroma_client = chromadb.PersistentClient(path=CHROMA_DB_PATH) 
collection = chroma_client.get_collection(name=CHROMA_DB_NAME)

In [ ]:
# Define function to retrieve top `k` chunks for every query [retrieve_top_k]
def retrieve_top_k(
    query_embeddings: np.ndarray | list[list[float]], 
    k: int = 10, 
    overfetch: int = 15, 
    batch_size: int = 50
) -> list[list[str]]:
    
    query_embeddings = np.asarray(query_embeddings)
    all_docs = []

    for start in tqdm(range(0, len(query_embeddings), batch_size), desc="Retrieving"):
        batch = query_embeddings[start:start + batch_size]
        results = collection.query(query_embeddings=batch.tolist(), n_results=k + overfetch)

        for docs in results["documents"]: # type: ignore
            seen = set()
            unique_docs = []
            
            for d in docs:
                if d not in seen:
                    seen.add(d)
                    unique_docs.append(d)
                    
                if len(unique_docs) == k:
                    break

            if len(unique_docs) < k:
                print(f"Warning: only {len(unique_docs)}/{k} unique chunks retrieved")

            all_docs.append(unique_docs)

    for i, docs in enumerate(all_docs):
        if len(docs) < k: print(f"Row {i} has {len(docs)} chunks, expected {k}")

    return all_docs

In [ ]:
# Get embeddings for train and test data
embedder = load_embedder(
    local_path=local_model_config["embedder_model"],
    hf_id=model_config["embedder_model"]
)

train_query_embeddings = embedder.encode(
    train_data["mcq_query"].to_list(), 
    show_progress_bar=True,
    batch_size=32
).astype(np.float32) # type: ignore

test_query_embeddings = embedder.encode(
    test_data["mcq_query"].to_list(), 
    show_progress_bar=True,
    batch_size=32
).astype(np.float32) # type: ignore

del embedder
reset_gpu()

In [ ]:
# Reduce high-dimensional embeddings to 10 dimensions using PaCMAP for clustering
pacmap = PaCMAP(
    n_components=10,
    n_neighbors=30,
    apply_pca=True,
    distance="euclidean",
    random_state=SEED
)

pacmap_data = pacmap.fit_transform(train_query_embeddings)

# Find clusters from 10 dimensional embeddings for better cross-validation
clusterer = HDBSCAN(
    min_cluster_size=7,
    min_samples=5,
    metric='euclidean',
    cluster_selection_method='eom',
    prediction_data=True
)

cluster_labels = clusterer.fit_predict(pacmap_data) # type: ignore

valid_clusters = np.array(cluster_labels)
noise_mask = valid_clusters == -1

if noise_mask.sum() > 0:
    knn = KNeighborsClassifier(n_neighbors=3)
    knn.fit(pacmap_data[~noise_mask], valid_clusters[~noise_mask]) # type: ignore
    
    # Predict the nearest cluster for noise points
    valid_clusters[noise_mask] = knn.predict(pacmap_data[noise_mask]) # type: ignore

# Check cluster information
n_clusters = len(set(valid_clusters))
print("Number of clusters:", n_clusters)

cluster_sizes = (
    pl.DataFrame({"cluster": valid_clusters})
    .group_by("cluster")
    .len()
    .sort("len", descending=True)
)
print(cluster_sizes)

In [ ]:
# Get retrieved chunks for train and test data
train_retrieved_chunks = retrieve_top_k(train_query_embeddings, k=10)
train_data = train_data.with_columns(
    pl.Series("retrieved_chunks", train_retrieved_chunks),
    pl.Series("cluster", valid_clusters)
)
train_data.write_parquet(inter_path_config["train_retrieved"])

test_retrieved_chunks = retrieve_top_k(test_query_embeddings, k=10)
test_data = test_data.with_columns(
    pl.Series("retrieved_chunks", test_retrieved_chunks)
)
test_data.write_parquet(inter_path_config["test_retrieved"])

In [ ]:
%%writefile rerank_worker.py
# Define script to parallelize ranking of retrieved documents
import gc, sys, torch, argparse

from tqdm.auto import tqdm

import numpy as np
import polars as pl

from pathlib import Path

from sentence_transformers import CrossEncoder
from transformers import BitsAndBytesConfig

def load_reranker(local_path: Path, hf_id: str, bnb_config: BitsAndBytesConfig, args):
    path_str = str(local_path)
    is_saved = (local_path / "config.json").exists() 
    model_name_or_path = path_str if is_saved else hf_id
    
    print(f"{'Loading' if is_saved else 'Downloading'} Reranker: {model_name_or_path}")
    
    model = CrossEncoder(
        model_name_or_path,
        trust_remote_code=True,
        model_kwargs={
            "quantization_config": bnb_config,
            "dtype": torch.float16,
            "device_map": "cuda:0"
        },
        prompts={"rerank": args.instruction},
        default_prompt_name="rerank"
    )
    
    if not is_saved:
        model.save(path_str)
        
    return model

def main() -> None:
    p = argparse.ArgumentParser()
    p.add_argument("--input", required=True)
    p.add_argument("--output", required=True)
    p.add_argument("--local_model_path", required=True)
    p.add_argument("--model_path", required=True)
    p.add_argument("--instruction", required=True)
    p.add_argument("--batch_size", type=int, default=16)
    p.add_argument("--k", type=int, default=5)
    args = p.parse_args()

    payload = pl.read_parquet(args.input)
    query_texts = payload["query"].to_list()
    retrieved_chunks = payload["retrieved_chunks"].to_list()

    bnb_config = BitsAndBytesConfig(
        load_in_8bit=True,
        llm_int8_threshold=6.0,
        llm_int8_has_fp16_weight=False
    )

    model = load_reranker(
        local_path=Path(args.local_model_path),
        hf_id=args.model_path,
        bnb_config=bnb_config,
        args=args
    )

    if model.tokenizer.pad_token is None:
        model.tokenizer.pad_token = model.tokenizer.eos_token
        
    model.model.config.pad_token_id = model.tokenizer.pad_token_id
    model.model.config.use_cache = False
    model.model.eval()

    flat_pairs, boundaries = [], []
    idx = 0

    for q, chunks in zip(query_texts, retrieved_chunks):
        chunks = [c if c.strip() else " " for c in chunks]
        flat_pairs.extend((q, c) for c in chunks)
        boundaries.append((idx, idx + len(chunks)))
        idx += len(chunks)

    flat_scores = model.predict(
        flat_pairs,
        batch_size=args.batch_size,
        show_progress_bar=True,
        convert_to_numpy=True
    )

    del model
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

    top_chunks_per_query = []
    for (start, end), chunks in zip(boundaries, retrieved_chunks):
        scores = flat_scores[start:end]
        top_idx = np.argsort(-scores)[:args.k]
        top_chunks_per_query.append([chunks[i] for i in top_idx])

    pl.DataFrame({"top_chunks": top_chunks_per_query}).write_parquet(args.output)

if __name__ == "__main__":
    main()

In [ ]:
# Define function to parallelize ranking of retrieved documents [rerank_top_k]
def rerank_top_k(query_texts, retrieved_chunks, tag, k=5, batch_size=16):
    mid = len(query_texts) // 2
    halves = {
        0: (query_texts[:mid], retrieved_chunks[:mid]),
        1: (query_texts[mid:], retrieved_chunks[mid:]),
    }

    input_paths, output_paths = {}, {}
    for gpu_id, (qs, chunks) in halves.items():
        input_paths[gpu_id] = f"{tag}_{gpu_id}_input.parquet"
        output_paths[gpu_id] = f"{tag}_{gpu_id}_output.parquet"

        pl.DataFrame({"query": qs, "retrieved_chunks": chunks}).write_parquet(input_paths[gpu_id])

    procs = []
    for gpu_id in (0, 1):
        env = {**os.environ, "CUDA_VISIBLE_DEVICES": str(gpu_id)}
        cmd = [
            sys.executable, "rerank_worker.py",
            "--input", input_paths[gpu_id],
            "--output", output_paths[gpu_id],
            "--local_model_path", local_model_config["reranker_model"],
            "--model_path", model_config["reranker_model"],
            "--instruction", MCQ_RERANK_INSTRUCTION,
            "--batch_size", str(batch_size),
            "--k", str(k)
        ]
        procs.append(subprocess.Popen(cmd, env=env))

    for proc in procs:
        if proc.wait() != 0:
            raise RuntimeError("rerank_worker.py failed - check the cell output above for the traceback")

    top0 = pl.read_parquet(output_paths[0])["top_chunks"].to_list()
    top1 = pl.read_parquet(output_paths[1])["top_chunks"].to_list()
    reset_gpu()
    
    return top0 + top1

In [ ]:
# Get reranked chunks from retrieved chunks for train and test data
train_retrieved_chunks = pl.read_parquet(inter_path_config["train_retrieved"])["retrieved_chunks"].to_list()
train_ranked_chunks = rerank_top_k(
    train_data["mcq_query"].to_list(), 
    train_retrieved_chunks, 
    tag="train"
)
train_data = train_data.with_columns(pl.Series("ranked_chunks", train_ranked_chunks))
train_data.write_parquet(inter_path_config["train_ranked"])

test_retrieved_chunks = pl.read_parquet(inter_path_config["test_retrieved"])["retrieved_chunks"].to_list()
test_ranked_chunks = rerank_top_k(
    test_data["mcq_query"].to_list(), 
    test_retrieved_chunks, 
    tag="test"
)
test_data = test_data.with_columns(pl.Series("ranked_chunks", test_ranked_chunks))
test_data.write_parquet(inter_path_config["test_ranked"])

In [ ]:
# Delete intermediate files from disk
patterns = ["*_input.parquet", "*_output.parquet", "*_retrieved.parquet"]

for pattern in patterns:
    for file_path in inter_path_config["working_dir"].glob(pattern):
        if file_path.is_file():
            file_path.unlink()
            print(f"Successfully deleted: {file_path.name}")

In [ ]:
# Load tokenizer and model for generative_slm on devices
reset_gpu()

tokenizer, model = load_slm(
    local_path=local_model_config["generative_slm"],
    hf_id=model_config["generative_slm"]
)

OPTION_TOKEN_IDS = {
    opt: tokenizer(f" {opt}", add_special_tokens=False)["input_ids"][-1]
    for opt in OPTION_COLS
}

In [ ]:
# Define function to build ranking prompt for SLM [build_ranking_prompt]
def build_ranking_prompt(row: dict, chunks: list[str]) -> str:
    context = "\n\n".join(chunks) if chunks else ""
    
    options_block = "\n".join(
        f"{opt}) {row[opt]}" for opt in OPTION_COLS if row[opt] is not None
    )

    user_content = f"Context:\n{context}\n\nQuestion: {row["prompt"]}\nOptions:\n{options_block}\n\nAnswer:"
    
    messages = [
        {"role": "system", "content": GENERATIVE_RANKING_INSTRUCTION},
        {"role": "user", "content": user_content}
    ]

    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

In [6]:
# Define function to build generation prompt for SLM [build_generation_prompt]
def build_generation_prompt(row: dict, chunks: list[str] | None) -> str:
    context = "\n\n".join(chunks) if chunks else ""

    options_block = "\n".join(
        f"{opt}) {row[opt]}" for opt in OPTION_COLS if row[opt] is not None
    )

    user_content = f"Context:\n{context}\n\nQuestion: {row["prompt"]}\nOptions:\n{options_block}\n\nAnswer:"

    messages = [
        {"role": "system", "content": GENERATIVE_SLM_INSTRUCTION},
        {"role": "user", "content": user_content}
    ]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

In [ ]:
# Define function to rank all 5 options [rank_options]
def rank_options(
    data: pl.DataFrame, 
    top_chunks: list[list[str]], 
    batch_size: int = 4
) -> list[str]:
    
    rows = data.to_dicts()
    predictions = []
    reset_gpu()

    ALL_OPT_IDS = torch.tensor([OPTION_TOKEN_IDS[o] for o in OPTION_COLS], device=model.device)
    
    with torch.inference_mode():
        for start in tqdm(range(0, len(rows), batch_size), desc="Scoring"):
            batch_rows = rows[start:start + batch_size]
            batch_chunks = top_chunks[start:start + batch_size]

            prompts = [build_ranking_prompt(r, c) for r, c in zip(batch_rows, batch_chunks)]
            enc = tokenizer(prompts, padding=True, truncation=True, return_tensors="pt").to(model.device)

            logits = model(**enc).logits[:, -1, :]
            batch_opt_logits = logits[:, ALL_OPT_IDS]

            for i, row in enumerate(batch_rows):
                valid_mask = torch.tensor([row[opt] is not None for opt in OPTION_COLS], device=logits.device)
                valid_opts = [opt for opt, keep in zip(OPTION_COLS, valid_mask.tolist()) if keep]

                row_logits = batch_opt_logits[i][valid_mask]
                ranked = [valid_opts[j] for j in torch.argsort(row_logits, descending=True).tolist()]
                predictions.append(" ".join(ranked))

            reset_gpu()

    return predictions

In [ ]:
# Define function to rank top 3 options [predict_top3]
def predict_top3(
    data: pl.DataFrame, 
    top_chunks: list[list[str]], 
    batch_size: int = 4
) -> list[str]:
    
    return [" ".join(r.split()[:3]) for r in rank_options(data, top_chunks, batch_size)]

In [8]:
# Define functions to generate top 3 options [generate_top3] [parse_generated_output] [generate_top3_predictions]
def generate_top3(
    data: pl.DataFrame,
    top_chunks: list[list[str]], 
    batch_size: int = 4,
    max_new_tokens: int = 8
) -> list[str]:

    rows = data.to_dicts()
    raw_outputs = []
    reset_gpu()

    with torch.inference_mode():
        for start in tqdm(range(0, len(rows), batch_size), desc="Generating"):
            batch_rows = rows[start:start + batch_size]
            batch_chunks = top_chunks[start:start + batch_size]

            prompts = [build_generation_prompt(r, c) for r, c in zip(batch_rows, batch_chunks)]

            enc = tokenizer(
                prompts, 
                padding=True, 
                truncation=True, 
                return_tensors="pt"
                ).to(model.device)

            out = model.generate( # type: ignore
                **enc, 
                pad_token_id=tokenizer.pad_token_id,
                max_new_tokens=max_new_tokens, 
                do_sample=False,
                num_beams=1
            ) 

            gen_output = out[:, enc["input_ids"].shape[1]:]
            raw_outputs.extend(tokenizer.batch_decode(gen_output, skip_special_tokens=True))
            reset_gpu()

    del batch_rows, batch_chunks, prompts, enc, out, gen_output # type: ignore
    reset_gpu()

    return raw_outputs

def parse_generated_output(raw: str, valid_opts: list[str]) -> str:
    seen = []

    for ch in raw.upper():
        if ch in valid_opts and ch not in seen:
            seen.append(ch)
        if len(seen) == 3:
            break

    for opt in valid_opts:
        if len(seen) == 3:
            break
        if opt not in seen:
            seen.append(opt)

    return " ".join(seen)

def generate_top3_predictions(
    data: pl.DataFrame,
    top_chunks: list[list[str]],
    batch_size: int = 4,
    max_new_tokens: int = 8
) -> list[str]:
    
    raw_outputs = generate_top3(data, top_chunks, batch_size=batch_size, max_new_tokens=max_new_tokens)
    rows = data.to_dicts()

    predictions = []
    for raw, row in zip(raw_outputs, rows):
        valid_opts = [opt for opt in OPTION_COLS if row[opt] is not None]
        predictions.append(parse_generated_output(raw, valid_opts))

    return predictions

In [9]:
# Define function to calculate AP@3
def average_precision_at_3(pred_letters: list[str], true_letter: str) -> float:

    for i, p in enumerate(pred_letters[:3]):
        if p == true_letter:
            return 1.0 / (i + 1)
        
    return 0.0

In [10]:
# Define function to compare RAG performance with zero-shot, and ranked data [run_full_pipeline_eval]
def run_full_pipeline_eval(
    data: pl.DataFrame, 
    ranked_chunks_list: list[list[str]],
    group_col: str, 
    batch_size: int = 4
) -> tuple[pl.DataFrame, pl.DataFrame]:
    
    true_letters = data["answer"].to_list()

    reset_gpu()
    zeroshot_preds = generate_top3_predictions(data, [[] for _ in range(len(data))], batch_size=batch_size)
    zeroshot_ap3 = [average_precision_at_3(p.split(), t) for p, t in zip(zeroshot_preds, true_letters)]

    reset_gpu()
    ranked_preds = generate_top3_predictions(data, ranked_chunks_list, batch_size=batch_size)
    ranked_ap3 = [average_precision_at_3(p.split(), t) for p, t in zip(ranked_preds, true_letters)]

    result = data.select(group_col).with_columns(
        pl.Series("zeroshot_pred", zeroshot_preds),
        pl.Series("zeroshot_ap3", zeroshot_ap3),
        pl.Series("ranked_pred", ranked_preds),
        pl.Series("ranked_ap3", ranked_ap3),
        pl.Series("answer", true_letters)
    )

    group_stats = (
        result.group_by(group_col)
        .agg(
            pl.col("zeroshot_ap3").mean().alias("zeroshot_map3"),
            pl.col("ranked_ap3").mean().alias("ranked_map3"), 
            pl.len().alias("n")
        )
        .with_columns(
            (pl.col("ranked_map3") - pl.col("zeroshot_map3")).alias("rag_diff")
            )
    )

    zeroshot_map3 = result["zeroshot_ap3"].mean()
    ranked_map3 = result["ranked_ap3"].mean()

    print(f"Zero-shot MAP@3: {zeroshot_map3:.8f}")
    print(f"Ranked MAP@3   : {ranked_map3:.8f}")

    print(f"\nRanked - Zero-shot MAP@3 Diff: {ranked_map3 - zeroshot_map3:+.8f}") # type: ignore
    
    return result, group_stats

In [ ]:
# # Compare RAG performance with zero-shot, and ranked data
# train_data = pl.read_parquet(inter_path_config["train_ranked"])

# train_samples = (
#     train_data.sample(fraction=1.0, shuffle=True, seed=SEED)
#     .group_by("cluster")
#     .head(5)
# )

# train_result, train_group_stats = run_full_pipeline_eval(
#     train_samples, 
#     ranked_chunks_list=train_samples["ranked_chunks"].to_list(), 
#     group_col="cluster", 
#     batch_size=5
# )

# # Ranked
# # Zero-shot MAP@3: 0.91239316
# # Ranked MAP@3   : 0.96816239

# # Ranked - Zero-shot MAP@3 Diff: +0.05576923

# # Generated
# # Zero-shot MAP@3: 0.89358974
# # Ranked MAP@3   : 0.95811966

# # Ranked - Zero-shot MAP@3 Diff: +0.06452991

Generating:   0%|          | 0/156 [00:00<?, ?it/s]

Generating:   0%|          | 0/156 [00:00<?, ?it/s]

Zero-shot MAP@3: 0.89358974
Ranked MAP@3   : 0.95811966

Ranked - Zero-shot MAP@3 Diff: +0.06452991


In [12]:
# Define function to check positional bias for options [positional_bias_check]
def positional_bias_check(preds: list[str], true_letters: list[str]) -> None:
    pred_top1 = [p.split()[0] for p in preds]
    pred_dist = Counter(pred_top1)
    true_dist = Counter(true_letters)
    
    n = len(preds)
    print(f"{'Letter':<8}{'Pred %':<10}{'True %':<10}{'Diff':<8}")
    
    for letter in OPTION_COLS:
        p = pred_dist.get(letter, 0) / n
        t = true_dist.get(letter, 0) / n
        
        print(f"{letter:<8}{p:<10.3f}{t:<10.3f}{p - t:+.3f}")

In [15]:
# positional_bias_check(train_result["ranked_pred"], train_result["answer"]) # type: ignore

# # Ranked
# # Letter  Pred %    True %    Diff    
# # A       0.178     0.183     -0.005
# # B       0.251     0.253     -0.001
# # C       0.188     0.190     -0.001
# # D       0.232     0.209     +0.023
# # E       0.150     0.165     -0.015

# # Generated
# # Letter  Pred %    True %    Diff    
# # A       0.178     0.183     -0.005
# # B       0.262     0.253     +0.009
# # C       0.192     0.190     +0.003
# # D       0.224     0.209     +0.015
# # E       0.144     0.165     -0.022

In [16]:
# Define function to check ranking of options [rank_calibration_check]
def rank_calibration_check(preds: list[str], true_letters: list[str]) -> None:
    pred_lists = [p.split() for p in preds]

    positions = [
        (p.index(t) + 1 if t in p else None)
        for p, t in zip(pred_lists, true_letters) if p[0] != t
    ]
    
    n = len(positions)
    rank2 = sum(p == 2 for p in positions)
    rank3 = sum(p == 3 for p in positions)
    missed = sum(p is None for p in positions)

    print(f"Among rows where top-1 was wrong ({n}/{len(preds)}):")
    print(f"\tcorrect answer at rank 2: {rank2}")
    print(f"\tcorrect answer at rank 3: {rank3}")
    print(f"\tcorrect answer outside top-3: {missed}")

In [18]:
# rank_calibration_check(train_result["ranked_pred"], train_result["answer"]) # type: ignore

# # Ranked
# # Among rows where top-1 was wrong (45/780):
# # 	correct answer at rank 2: 35
# # 	correct answer at rank 3: 8
# # 	correct answer outside top-3: 2

# # Generated
# # Among rows where top-1 was wrong (53/780):
# # 	correct answer at rank 2: 36
# # 	correct answer at rank 3: 7
# # 	correct answer outside top-3: 10

In [ ]:
# train_ranked_chunks = pl.read_parquet(inter_path_config["train_ranked"])["ranked_chunks"].to_list()
# train_data = train_data.with_columns(pl.Series("top3_pred", predict_top3(train_ranked_chunks, train_ranked)))
# train_data.write_parquet(inter_path_config["train_final"])

test_data = pl.read_parquet(inter_path_config["test_ranked"])
test_ranked_chunks = test_data["ranked_chunks"].to_list()
test_data = test_data.with_columns(pl.Series("top3_pred", predict_top3(test_data, test_ranked_chunks)))

test_data.write_parquet(inter_path_config["test_final"])

Scoring:   0%|          | 0/125 [00:00<?, ?it/s]

In [16]:
submission = test_data.select(["id", "top3_pred"]).rename({"id": "ID", "top3_pred": "Prediction"})
submission.write_csv(inter_path_config["submission"])